# 6BUIS017W CWK1 - Task 1: Data preparation and feature engineering
Run top to bottom (Runtime > Run all). Files are saved in `data/`.

| Section | Brief item |
|---|---|
| 1 | 1.1 tickers + data acquisition |
| 2 | 1.2 validation, exclusion log |
| 3 | 1.3 weekly return and beta |
| 4 | 1.4 R, F, M + worked example |
| 5 | 1.5 summaries and feature table |

In [ ]:
# 0. Setup and versions (the brief asks for package versions)
import sys, io, json, datetime as dt
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError
import numpy as np, pandas as pd, requests

for pkg in ["pandas", "numpy", "yfinance", "requests", "lxml", "scipy", "scikit-learn", "matplotlib"]:
    try: print(f"{pkg:14s}{version(pkg)}")
    except PackageNotFoundError: print(f"{pkg:14s}not installed")
print("python        ", sys.version.split()[0])

DATA_DIR = Path("data"); DATA_DIR.mkdir(exist_ok=True)
START, END = "2021-01-01", "2025-01-01"      # END is exclusive (brief 1.1)
BENCH = "^GSPC"
BASE_WIN, MULT, RFM_WIN = 20, 2, 60           # 20-session baseline, 2x rule, 60-session window

pandas        2.2.3
numpy         2.4.6
yfinance      0.2.66
requests      2.32.4
lxml          6.1.3
scipy         1.16.3
scikit-learn  1.6.1
matplotlib    3.10.0
python         3.13.18


## 1. Ticker selection and data acquisition (1.1)
Tickers come from Wikipedia via pandas. Prices come from **yfinance** (free, no API key, same source as the tutorials).
If cached files already exist in `data/` they are loaded instead of downloading again - this is the "documented cached dataset" the brief allows, and it makes the notebook reproducible.

In [ ]:
CONS, META = DATA_DIR / "sp500_constituents.csv", DATA_DIR / "retrieval_metadata.json"
meta = json.load(open(META)) if META.exists() else {}

# The ticker list is the latest Wikipedia revision on or before 2 November 2026 (revision cutoff).
# The REAL retrieval date (the day this code runs) is recorded separately - never overwrite it.
SNAPSHOT_CUTOFF = "2026-11-02T23:59:59Z"
WIKI_TITLE = "List of S&P 500 companies"
HDR = {"User-Agent": "Mozilla/5.0"}                      # Wikipedia blocks the default agent

if CONS.exists() and "constituents_retrieved" in meta:
    constituents = pd.read_csv(CONS)
    print("Loaded cached constituent list. Retrieved on", meta["constituents_retrieved"],
          "| Wikipedia revision", meta.get("wikipedia_revision_id"), "dated", meta.get("wikipedia_revision_timestamp"))
else:
    api = requests.get("https://en.wikipedia.org/w/api.php", headers=HDR, timeout=30, params=dict(
        action="query", prop="revisions", titles=WIKI_TITLE, rvlimit=1, rvstart=SNAPSHOT_CUTOFF,
        rvdir="older", rvprop="ids|timestamp", format="json")).json()
    rev = next(iter(api["query"]["pages"].values()))["revisions"][0]      # last revision on/before 2 Nov 2026
    url = f"https://en.wikipedia.org/w/index.php?title=List_of_S%26P_500_companies&oldid={rev['revid']}"
    html = requests.get(url, headers=HDR, timeout=30).text
    constituents = pd.read_html(io.StringIO(html), match="Symbol")[0]      # pandas extraction, as required
    constituents["Symbol"] = constituents["Symbol"].str.replace(".", "-", regex=False)   # BRK.B -> BRK-B (Yahoo format)
    constituents.to_csv(CONS, index=False)
    meta.update(constituents_source=url,
                wikipedia_revision_id=rev["revid"], wikipedia_revision_timestamp=rev["timestamp"],
                constituents_snapshot_target="2026-11-02",
                constituents_retrieved=dt.date.today().isoformat(),      # real date this code was run
                rerun_note="Earlier outputs were removed and the full pipeline was re-run; "
                           "all results come from the retrieval date above.")
    json.dump(meta, open(META, "w"), indent=2)

print("Tickers on list:", len(constituents), "| unique:", constituents["Symbol"].nunique())
print("Retrieval date (real run):", meta["constituents_retrieved"], "| list = Wikipedia revision of", meta["wikipedia_revision_timestamp"])
constituents[["Symbol", "Security", "GICS Sector"]].head()


Tickers on list: 503 | unique: 503
Retrieval date (real run): 2026-11-19 | list = Wikipedia revision of 2026-10-27T02:50:28Z


,Symbol,Security,GICS Sector
0,MMM,3M,Industrials
1,AOS,A. O. Smith,Industrials
2,ABT,Abbott Laboratories,Health Care
3,ABBV,AbbVie,Health Care
4,ACN,Accenture,Information Technology


In [ ]:
PRICE_FILES = {k: DATA_DIR / f"{k}.csv" for k in ["adj_close", "close", "volume"]}

def pick(df, field, t):
    s = df[field]
    return s[t] if isinstance(s, pd.DataFrame) else s

if all(p.exists() for p in PRICE_FILES.values()):
    adj, close, vol = [pd.read_csv(PRICE_FILES[k], index_col=0, parse_dates=True) for k in ["adj_close", "close", "volume"]]
    print("Loaded cached price snapshot; downloaded on", meta.get("prices_downloaded", "unknown date"))
else:
    import yfinance as yf
    tickers = constituents["Symbol"].tolist() + [BENCH]
    # auto_adjust=False keeps BOTH Close and Adj Close
    raw = yf.download(tickers, start=START, end=END, auto_adjust=False, progress=False, threads=True)
    adj, close, vol = raw["Adj Close"].copy(), raw["Close"].copy(), raw["Volume"].copy()
    failed = [t for t in tickers if adj[t].isna().all()]
    if failed:                                   # one retry for tickers that returned nothing (rate limits)
        print("Retrying", len(failed), "tickers:", failed)
        r2 = yf.download(failed, start=START, end=END, auto_adjust=False, progress=False)
        for t in failed:
            adj[t], close[t], vol[t] = pick(r2, "Adj Close", t), pick(r2, "Close", t), pick(r2, "Volume", t)
    for df, k in [(adj, "adj_close"), (close, "close"), (vol, "volume")]:
        df.to_csv(PRICE_FILES[k])
    meta.update(prices_downloaded=dt.date.today().isoformat(), price_source="yfinance " + version("yfinance"),
                download_args="auto_adjust=False, start=2021-01-01, end=2025-01-01 (exclusive)")
    json.dump(meta, open(META, "w"), indent=2)

for df in (adj, close, vol):
    df.index = pd.to_datetime(df.index).tz_localize(None) if df.index.tz is not None else pd.to_datetime(df.index)
print("Price matrix (dates x tickers):", adj.shape, "| range:", adj.index.min().date(), "to", adj.index.max().date())
print(json.dumps(meta, indent=2))

ERROR:yfinance:
6 Failed downloads:
ERROR:yfinance:['SNDK', 'Q', 'FDXF', 'KVUE', 'HONA', 'VYLR']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')


Retrying 6 tickers: ['FDXF', 'HONA', 'KVUE', 'Q', 'SNDK', 'VYLR']


ERROR:yfinance:
6 Failed downloads:
ERROR:yfinance:['KVUE', 'Q', 'HONA', 'VYLR', 'SNDK', 'FDXF']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')


Price matrix (dates x tickers): (1005, 504) | range: 2021-01-04 to 2024-12-31
{
  "constituents_source": "https://en.wikipedia.org/w/index.php?title=List_of_S%26P_500_companies&oldid=1382178819",
  "wikipedia_revision_id": 1382178819,
  "wikipedia_revision_timestamp": "2026-10-27T02:50:28Z",
  "constituents_snapshot_target": "2026-11-02",
  "constituents_retrieved": "2026-11-19",
  "rerun_note": "Earlier outputs were removed and the full pipeline was re-run; all results come from the retrieval date above.",
  "prices_downloaded": "2026-11-19",
  "price_source": "yfinance 0.2.66",
  "download_args": "auto_adjust=False, start=2021-01-01, end=2025-01-01 (exclusive)"
}


## 2. Data validation and alignment (1.2)
Rules (no silent filling anywhere): a share is **kept** only if, on every benchmark trading day, it has Adj Close, Close and Volume, all prices are positive, and volume is positive.
Every exclusion goes into the log with its reason. Corporate actions are *checked and reported* (not repaired).

In [ ]:
# Duplicate dates and the common trading calendar
for name, df in [("adj_close", adj), ("close", close), ("volume", vol)]:
    print(f"{name:10s} duplicate dates: {df.index.duplicated().sum()} | sorted: {df.index.is_monotonic_increasing}")

bench_dates = adj[BENCH].dropna().index
print("Benchmark trading days:", len(bench_dates), "|", bench_dates[0].date(), "->", bench_dates[-1].date())
dropped_rows = len(adj.index.difference(bench_dates))
print("Rows not on the benchmark calendar (removed):", dropped_rows)
adj, close, vol = adj.loc[bench_dates], close.loc[bench_dates], vol.loc[bench_dates]
assert adj[BENCH].notna().all() and (adj[BENCH] > 0).all(), "benchmark must be clean"

adj_close  duplicate dates: 0 | sorted: True
close      duplicate dates: 0 | sorted: True
volume     duplicate dates: 0 | sorted: True
Benchmark trading days: 1005 | 2021-01-04 -> 2024-12-31
Rows not on the benchmark calendar (removed): 0


In [ ]:
shares = [t for t in constituents["Symbol"] if t != BENCH]
exclusions = []        # (ticker, reason)
rows = []
for t in shares:
    if t not in adj.columns or adj[t].isna().all():
        exclusions.append((t, "no price data returned by yfinance")); continue
    a, c, v = adj[t], close[t], vol[t]
    n_miss = int(a.isna().sum() + c.isna().sum() + v.isna().sum())
    first_valid = a.first_valid_index()
    nonpos = int(((a <= 0) | (c <= 0)).sum())
    negvol = int((v < 0).sum())
    zerovol = int((v == 0).sum())
    ret_c, ret_a = c.pct_change(fill_method=None), a.pct_change(fill_method=None)
    rows.append(dict(Ticker=t, first_valid=first_valid, n_missing=n_miss, nonpositive=nonpos, neg_vol=negvol,
                     zero_vol=zerovol, max_abs_close_ret=ret_c.abs().max(),
                     big_moves_gt40pct=int((ret_c.abs() > 0.40).sum()),
                     max_adj_vs_close_ret_gap=(ret_a - ret_c).abs().max()))
    if first_valid is not None and first_valid > bench_dates[0]:
        exclusions.append((t, f"unequal history: first price {first_valid.date()} (listed after study start)"))
    elif n_miss > 0:
        exclusions.append((t, f"{n_miss} missing Adj Close/Close/Volume values"))
    if nonpos: exclusions.append((t, f"{nonpos} non-positive prices"))
    if negvol: exclusions.append((t, f"{negvol} negative volume values"))
    if zerovol: exclusions.append((t, f"{zerovol} zero-volume days"))

checks = pd.DataFrame(rows).set_index("Ticker")
excl_log = pd.DataFrame(exclusions, columns=["Ticker", "Reason"])
valid = [t for t in shares if t not in set(excl_log["Ticker"])]
excl_log.to_csv(DATA_DIR / "exclusion_log_task1.csv", index=False)

print(f"Shares on list: {len(shares)} | kept: {len(valid)} | excluded: {excl_log['Ticker'].nunique()}")
print(excl_log["Reason"].str.replace(r"\d+", "N", regex=True).str.replace(r"first price .*? \(", "first price D (", regex=True)
      .value_counts().to_string())
excl_log

Shares on list: 503 | kept: 485 | excluded: 18
Reason
unequal history: first price D (listed after study start)    10
no price data returned by yfinance                            6
N zero-volume days                                            2


,Ticker,Reason
0,APP,unequal history: first price 2021-04-15 (liste...
1,COIN,unequal history: first price 2021-04-14 (liste...
2,CEG,unequal history: first price 2022-01-19 (liste...
3,EXE,unequal history: first price 2021-02-10 (liste...
4,FDXF,no price data returned by yfinance
5,FERG,27 zero-volume days
6,GEHC,unequal history: first price 2022-12-15 (liste...
7,GEV,unequal history: first price 2024-03-27 (liste...
8,HONA,no price data returned by yfinance
9,KVUE,no price data returned by yfinance


In [ ]:
# Corporate-action check. Yahoo's Close is split-adjusted; Adj Close additionally adjusts for dividends.
# So Close x Volume stays split-compatible. We verify this by looking for split-sized jumps in Close
# and for adjusted-vs-close return gaps that are too large to be an ordinary dividend.
v = checks.loc[valid]
print("Shares with any daily Close move > 40%:", int((v["big_moves_gt40pct"] > 0).sum()))
print("Largest daily |Close return| (top 5):\n", v["max_abs_close_ret"].sort_values(ascending=False).head(5).round(3).to_string())
print("\nLargest adj-vs-close daily return gap (top 5; dividends only should be small):\n",
      v["max_adj_vs_close_ret_gap"].sort_values(ascending=False).head(5).round(4).to_string())
# Shares with a gap > 10% need a manual look (possible split not applied consistently in one series)
flag = v.index[v["max_adj_vs_close_ret_gap"] > 0.10].tolist()
print("\nFlagged for manual inspection (gap > 10%):", flag)

Shares with any daily Close move > 40%: 4
Largest daily |Close return| (top 5):
 Ticker
BE      0.592
CVNA    0.560
GL      0.531
DXCM    0.407
BIIB    0.399

Largest adj-vs-close daily return gap (top 5; dividends only should be small):
 Ticker
F       0.0595
LYB     0.0547
TDG     0.0533
ACGL    0.0492
PGR     0.0468

Flagged for manual inspection (gap > 10%): []


**Note:** the constituent list is a 2026 snapshot applied to 2021-2024 prices, so companies removed from the index are absent and later additions are included (survivorship bias).

## 3. Weekly return and weekly beta (1.3)
Friday-ending weeks (`W-FRI`), last available adjusted close in each week. Incomplete first/last calendar weeks are dropped; holiday-shortened weeks are kept.

In [ ]:
adj_v = adj[valid + [BENCH]]
wk = adj_v.resample("W-FRI").last()                       # label = Friday; value = last trading day's Adj Close in that week
n_trading_days = adj_v.resample("W-FRI").count()[BENCH]   # trading days inside each weekly bin
last_day = bench_dates[-1]

# Week is complete if its Monday is on/after START and its Friday label is on/before the last data day
complete = ((wk.index - pd.Timedelta(days=4)) >= pd.Timestamp(START)) & (wk.index <= last_day)
print("Weekly bins:", len(wk), "| dropped as incomplete:", [d.date() for d in wk.index[~complete]])
print("Shortest retained weeks (trading days):", n_trading_days[complete].value_counts().sort_index().to_dict())

weekly = wk.loc[complete]
assert weekly.notna().all().all(), "unexpected missing weekly prices (would need documenting, not filling)"
rets = weekly.pct_change(fill_method=None).iloc[1:]       # return = P_t / P_(t-1) - 1; first week has no predecessor
assert rets.notna().all().all()
N_OBS = len(rets)
print(f"\nAligned weekly return observations per share: {N_OBS}  ({rets.index[0].date()} -> {rets.index[-1].date()})")
print("Benchmark and shares share identical week labels:", rets[BENCH].index.equals(rets[valid].index))

Weekly bins: 209 | dropped as incomplete: [datetime.date(2025, 1, 3)]
Shortest retained weeks (trading days): {4: 37, 5: 171}

Aligned weekly return observations per share: 207  (2021-01-15 -> 2024-12-27)
Benchmark and shares share identical week labels: True


In [ ]:
bench_r = rets[BENCH]
mean_ret = rets[valid].mean()                                         # arithmetic mean weekly return (decimal)
beta = rets[valid].apply(lambda s: s.cov(bench_r)) / bench_r.var()    # cov(share, market) / var(market), both ddof=1

# Independent checks:
t0 = valid[0]
slope = np.polyfit(bench_r.values, rets[t0].values, 1)[0]
print(f"Check beta({t0}): formula {beta[t0]:.6f} vs regression slope {slope:.6f}")
assert np.allclose(beta[t0], slope)
p = weekly[t0].iloc[-2:]
print(f"Check return({t0}) last week: {p.iloc[1]:.4f}/{p.iloc[0]:.4f}-1 = {p.iloc[1]/p.iloc[0]-1:.6f} vs {rets[t0].iloc[-1]:.6f}")

# Why Adj Close: compare with Close-based weekly returns (the gap is mostly dividends)
close_r = close[valid].resample("W-FRI").last().loc[complete].pct_change(fill_method=None).iloc[1:]
gap = (rets[valid].mean() - close_r.mean())
print(f"Mean weekly return, Adj minus Close basis: avg {gap.mean():.5f}, max {gap.max():.5f} (positive = dividends)")
print(f"\nBenchmark mean weekly return {bench_r.mean():.5f}, variance {bench_r.var():.6f}")
print("Beta summary:\n", beta.describe().round(3).to_string())

Check beta(MMM): formula 0.869079 vs regression slope 0.869079
Check return(MMM) last week: 125.4633/124.5960-1 = 0.006961 vs 0.006961
Mean weekly return, Adj minus Close basis: avg 0.00035, max 0.00152 (positive = dividends)

Benchmark mean weekly return 0.00240, variance 0.000492
Beta summary:
 count    485.000
mean       0.972
std        0.406
min        0.055
25%        0.667
50%        0.948
75%        1.179
max        3.779


## 4. Adapted RFM (1.4)
Observation date = the final common completed trading date. An *event* is a day whose volume is **greater than** 2 x the mean volume of the **previous** 20 sessions (the day itself is excluded via `shift(1)`).
- **R** = calendar days from the latest event (anywhere in the valid history) to the observation date. No event -> missing, not 0.
- **F** = number of event days in the last 60 sessions.
- **M** = sum of Close x Volume over the last 60 sessions (approximate USD traded value).

In [ ]:
vol_v, close_v = vol[valid], close[valid]
obs_date = vol_v.index.max()
assert (vol_v.loc[obs_date].notna().all()) and obs_date == bench_dates[-1]
print("Observation date:", obs_date.date(), "| sessions available:", len(vol_v))

prior_mean = vol_v.rolling(BASE_WIN).mean().shift(1)     # mean of the 20 sessions BEFORE each day (day excluded)
threshold = MULT * prior_mean                            # per share, per day
event = vol_v > threshold                                # strict ">" ; NaN threshold (first 20 days) -> False
print("Events flagged in total:", int(event.sum().sum()), "| first day with a valid threshold:", threshold.dropna(how="all").index[0].date())

# Recency (calendar days; weekends/holidays count). No event -> NaN
last_event = event.apply(lambda s: s.index[s.values][-1] if s.any() else pd.NaT)
R = (obs_date - last_event).dt.days.astype("float")        # NaT -> NaN automatically
# Frequency and Monetary use only the latest 60 sessions
win = vol_v.index[-RFM_WIN:]
F = event.loc[win].sum()
M = (close_v.loc[win] * vol_v.loc[win]).sum()
print(f"60-session window: {win[0].date()} -> {win[-1].date()} ({len(win)} sessions)")
print("Shares with no event in valid history (R missing):", int(R.isna().sum()), "| shares with R = 0:", int((R == 0).sum()))

Observation date: 2024-12-31 | sessions available: 1005
Events flagged in total: 18222 | first day with a valid threshold: 2021-02-02
60-session window: 2024-10-07 -> 2024-12-31 (60 sessions)
Shares with no event in valid history (R missing): 0 | shares with R = 0: 0


In [ ]:
# Verification checks (no look-ahead, threshold alignment, boundary cases)
rng = np.random.default_rng(42)
for t in rng.choice(valid, 3, replace=False):
    i = int(rng.integers(BASE_WIN + 5, len(vol_v)))
    manual = MULT * vol_v[t].iloc[i - BASE_WIN:i].mean()        # sessions i-20 ... i-1, NOT including i
    assert np.isclose(manual, threshold[t].iloc[i]), (t, i)
    assert bool(event[t].iloc[i]) == bool(vol_v[t].iloc[i] > manual)
print("Threshold alignment verified for 3 random share-days (uses previous 20 sessions only).")
assert len(vol_v) >= BASE_WIN + RFM_WIN, "not enough history for 20 + 60"
print("History sufficient for the 20-session baseline AND the 60-session window.")
print("Events on the observation date:", int(event.loc[obs_date].sum()), "(these have R = 0)")

Threshold alignment verified for 3 random share-days (uses previous 20 sessions only).
History sufficient for the 20-session baseline AND the 60-session window.
Events on the observation date: 0 (these have R = 0)


In [ ]:
# Worked example for one share
ex = "AAPL" if "AAPL" in valid else valid[0]
tbl = pd.DataFrame({"Volume": vol_v[ex], "Prior20_mean": prior_mean[ex], "Threshold(2x)": threshold[ex],
                    "High_volume?": event[ex], "Close": close_v[ex], "Close x Volume": close_v[ex] * vol_v[ex]})
print("Worked example:", ex)
display(tbl.tail(8).round(2)) if "display" in globals() else print(tbl.tail(8).round(2))
print(f"\nObservation date {obs_date.date()} | latest event {last_event[ex].date() if pd.notna(last_event[ex]) else 'none'}")
print(f"R = {R[ex]:.0f} calendar days")
print(f"F = {int(F[ex])} event days in {win[0].date()} -> {win[-1].date()}")
print(f"M = sum(Close x Volume over 60 sessions) = ${M[ex]:,.0f}  (approx traded value, NOT market cap)")

Worked example: AAPL
               Volume  Prior20_mean  Threshold(2x)  High_volume?   Close  \
Date                                                                       
2024-12-19   60882300    43718970.0     87437940.0         False  249.79   
2024-12-20  147495300    45004605.0     90009210.0          True  254.49   
2024-12-23   40858800    50273955.0    100547910.0         False  255.27   
2024-12-24   23234700    50408480.0    100816960.0         False  258.20   
2024-12-26   27237100    47062575.0     94125150.0         False  259.02   
2024-12-27   42355300    46125120.0     92250240.0         False  255.59   
2024-12-30   35557500    46567965.0     93135930.0         False  252.20   
2024-12-31   39480700    46921770.0     93843540.0         False  250.42   

            Close x Volume  
Date                        
2024-12-19    1.520779e+10  
2024-12-20    3.753608e+10  
2024-12-23    1.043003e+10  
2024-12-24    5.999200e+09  
2024-12-26    7.054953e+09  
2024-12-27    1

## 5. Feature table, summaries and common share set (1.5)

In [ ]:
features = pd.DataFrame({"Mean_Weekly_Return": mean_ret, "Weekly_Beta": beta, "Weekly_Obs": N_OBS,
                         "R": R, "F": F.astype(int), "M": M, "Last_Event_Date": last_event})
features = constituents.set_index("Symbol")[["Security", "GICS Sector"]].join(features, how="inner")
features.index.name = "Ticker"

# Common valid share set: shares with all five features (R missing = no event -> excluded from RFM clustering)
no_event = features.index[features["R"].isna()].tolist()
common = features.dropna(subset=["Mean_Weekly_Return", "Weekly_Beta", "R", "F", "M"])
print(f"Shares with weekly features: {len(features)} | common set for ALL clustering: {len(common)}")
print("Excluded for no high-volume event (R missing):", no_event)
if no_event:
    pd.concat([excl_log, pd.DataFrame({"Ticker": no_event, "Reason": "no high-volume event in valid history (R missing)"})]
              ).to_csv(DATA_DIR / "exclusion_log_task1.csv", index=False)

common.to_csv(DATA_DIR / "features_task1.csv")
weekly.to_csv(DATA_DIR / "weekly_adj_close.csv"); rets.to_csv(DATA_DIR / "weekly_returns.csv")
event.to_csv(DATA_DIR / "high_volume_event_flags.csv")
common.head()

Shares with weekly features: 485 | common set for ALL clustering: 485
Excluded for no high-volume event (R missing): []


,Security,GICS Sector,Mean_Weekly_Return,Weekly_Beta,Weekly_Obs,R,F,M,Last_Event_Date
Ticker,,,,,,,,,
MMM,3M,Industrials,0.001142,0.869079,207,11.0,4,2.599146e+10,2024-12-20
AOS,A. O. Smith,Industrials,0.001991,1.042804,207,11.0,2,4.748980e+09,2024-12-20
ABT,Abbott Laboratories,Health Care,0.000971,0.583543,207,11.0,1,3.523027e+10,2024-12-20
ABBV,AbbVie,Health Care,0.003749,0.425366,207,11.0,3,6.837054e+10,2024-12-20
ACN,Accenture,Information Technology,0.002376,1.143300,207,11.0,2,5.199430e+10,2024-12-20


In [ ]:
num = common[["Mean_Weekly_Return", "Weekly_Beta", "R", "F", "M"]]
print(num.describe().round(4).to_string())
print("\nSkewness (raw):\n", num.skew().round(2).to_string())
print("Skewness of log1p(M):", round(float(np.log1p(common["M"]).skew()), 2))
print("\nCorrelation preview (raw):\n", num.corr().round(2).to_string())

       Mean_Weekly_Return  Weekly_Beta         R         F             M
count            485.0000     485.0000  485.0000  485.0000  4.850000e+02
mean               0.0029       0.9720   20.3938    2.9856  3.945567e+10
std                0.0026       0.4059   21.3624    1.3290  1.292672e+11
min               -0.0045       0.0555    4.0000    0.0000  1.297483e+09
25%                0.0012       0.6666   11.0000    2.0000  1.017679e+10
50%                0.0026       0.9480   11.0000    3.0000  1.670905e+10
75%                0.0042       1.1792   14.0000    4.0000  3.236510e+10
max                0.0178       3.7786  222.0000    8.0000  1.847697e+12

Skewness (raw):
 Mean_Weekly_Return     0.88
Weekly_Beta            1.11
R                      3.64
F                      0.43
M                     11.67
Skewness of log1p(M): 1.08

Correlation preview (raw):
                     Mean_Weekly_Return  Weekly_Beta     R     F     M
Mean_Weekly_Return                1.00         0.26  0.08  

In [ ]:
# Provenance for the data mart / reproducibility section (3.1.2): file hashes and package versions
import hashlib
meta.update(file_sha256={p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(DATA_DIR.glob("*.csv"))},
            packages={p: version(p) for p in ["pandas", "numpy", "yfinance", "scipy", "scikit-learn", "matplotlib"]},
            observation_date=str(obs_date.date()), rfm_window=f"{win[0].date()} to {win[-1].date()}",
            n_weekly_obs=int(N_OBS), n_shares_valid=len(valid), n_shares_clustered=len(common))
json.dump(meta, open(META, "w"), indent=2)
print(json.dumps(meta, indent=2))


{
  "constituents_source": "https://en.wikipedia.org/w/index.php?title=List_of_S%26P_500_companies&oldid=1382178819",
  "wikipedia_revision_id": 1382178819,
  "wikipedia_revision_timestamp": "2026-10-27T02:50:28Z",
  "constituents_snapshot_target": "2026-11-02",
  "constituents_retrieved": "2026-11-19",
  "rerun_note": "Earlier outputs were removed and the full pipeline was re-run; all results come from the retrieval date above.",
  "prices_downloaded": "2026-11-19",
  "price_source": "yfinance 0.2.66",
  "download_args": "auto_adjust=False, start=2021-01-01, end=2025-01-01 (exclusive)",
  "file_sha256": {
    "adj_close.csv": "702d41b1657f6e4879d4cb13b3ee2db4d8c9b51eddd4a04be157d9635c428d52",
    "close.csv": "d57e595a1dc06d223336009fabd7ba4631e88605c1db4def3c2c912f1a872536",
    "exclusion_log_task1.csv": "80502bbfa381aa3100e1ad22dd523d0b29369a70b17f35ddbd80175bf67ce155",
    "features_task1.csv": "27c90e685df857e3c0b052ef4c5979d30b313bbbf875b08e2141c630d670759c",
    "high_volume_ev

In [ ]:
# Optional: bundle everything for download (Colab only)
try:
    import shutil
    from google.colab import files
    shutil.make_archive("task1_outputs", "zip", "data")
    files.download("task1_outputs.zip")
except ImportError:
    print("Not in Colab - files are in the data/ folder.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>